<a href="https://colab.research.google.com/github/SaraEstilaei/incident-sentiment-analyzer/blob/Model%2FFinding-our-nlp-models(K)/Finding_Our_NLP_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate scikit-learn gdown

In [2]:
import os
import re
import inspect
import numpy as np
import pandas as pd
import torch
import gdown
import gc


from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

In [3]:
drive.mount("/content/drive")      # فقط برای ذخیره‌ی مدل روی Drive

SEED       = 42
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN    = 128
DATA_PATH  = "/content/sample_5k.csv"
SAVE_ROOT  = "/content/drive/MyDrive/NLP_Proj"
EPOCHS = 3
lr = 2e-5


print("GPU:", torch.cuda.is_available())

Mounted at /content/drive
GPU: True


In [4]:
import gdown

FILE_ID   = "1cPBbdUaIXE4LXHMm1euo8SsArHtt0F9i"
gdown.download(id=FILE_ID, output=DATA_PATH, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1cPBbdUaIXE4LXHMm1euo8SsArHtt0F9i
To: /content/sample_5k.csv
100%|██████████| 10.1M/10.1M [00:00<00:00, 92.7MB/s]


'/content/sample_5k.csv'

In [5]:
df = pd.read_csv(DATA_PATH, sep=None, engine="python")
print(df.shape)
print(df.columns.tolist())
print(df[["overall", "vote", "full_text"]].head())

(5000, 14)
['overall', 'vote', 'verified', 'reviewTime', 'reviewerID', 'asin', 'style', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime', 'review_year', 'reviewText_clean', 'full_text']
   overall  vote                                          full_text
0        5     2  Strong and secure. This is a very well made ba...
1        5     0  BUY IT!! This product is the best case I have ...
2        5     0  No issues, solid cables! I bought these to rep...
3        2     0  Good sound, poor bluetooth signal. It has a po...
4        5     0  Nice slender case The way the sleeves are atta...


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   overall           5000 non-null   int64 
 1   vote              5000 non-null   int64 
 2   verified          5000 non-null   bool  
 3   reviewTime        5000 non-null   object
 4   reviewerID        5000 non-null   object
 5   asin              5000 non-null   object
 6   style             2981 non-null   object
 7   reviewerName      5000 non-null   object
 8   reviewText        5000 non-null   object
 9   summary           5000 non-null   object
 10  unixReviewTime    5000 non-null   int64 
 11  review_year       5000 non-null   int64 
 12  reviewText_clean  5000 non-null   object
 13  full_text         5000 non-null   object
dtypes: bool(1), int64(4), object(9)
memory usage: 512.8+ KB


In [ ]:
print((df["full_text"].str.strip() == "").sum())

0


In [ ]:
# امتیاز 1 تا 5 رو تبدیل به 0 تا 4 میکنیم
df["label"] = df["overall"] - 1

#چون مدلمون عدد درک نمیکنه، تگ کوچیک برای مقادیر مختلف میذاریم
def vote_tag(n):
    return "votes none" if n == 0 else ("votes few" if n < 5 else "votes many")

print("توزیع امتیازها:")
print(df["overall"].value_counts(normalize=True).sort_index())

print("\nتوزیع vote:")
print(df["vote"].describe())
print("درصد نظرهای دارای vote:", (df["vote"] > 0).mean())

print("\nدرصد نظرهای دارای vote به تفکیک امتیاز:")
print(df.groupby("overall")["vote"].apply(lambda s: (s > 0).mean()))

train_df, val_df = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=SEED)
print("\n", train_df.shape, val_df.shape)

توزیع امتیازها:
overall
1    0.0988
2    0.0676
3    0.0968
4    0.1866
5    0.5502
Name: proportion, dtype: float64

توزیع vote:
count    5000.000000
mean        1.753800
std        12.153811
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       463.000000
Name: vote, dtype: float64
درصد نظرهای دارای vote: 0.2256

درصد نظرهای دارای vote به تفکیک امتیاز:
overall
1    0.342105
2    0.254438
3    0.214876
4    0.216506
5    0.206107
Name: vote, dtype: float64

 (4250, 15) (750, 15)


In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def build_dataset(d, use_vote=False, use_weights=False, train=True):
    """
    use_vote    : برچسب votes none/few/many به ابتدای متن اضافه می‌شود
    use_weights : ستون weight (برای وزن‌دهی loss) اضافه میشه و فقط روی train وزن‌داره
    """
    text = d["vote"].apply(vote_tag) + " . " + d["full_text"] if use_vote else d["full_text"]
    tmp = pd.DataFrame({"text": text.values, "label": d["label"].values})

    if use_weights:
        if train:
            w = 1 + 0.5 * np.log1p(d["vote"].values)
            w = np.clip(w, 1.0, 3.0)
            w = w / w.mean()
        else:
            w = np.ones(len(d)) # بدون وزن برای ولیدیشن
        tmp["weight"] = w.astype("float32")

    ds = Dataset.from_pandas(tmp, preserve_index=False)
    ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True)
    return ds.remove_columns(["text"])

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {"micro_f1": f1_score(p.label_ids, preds, average="micro"),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
class WeightedTrainer(Trainer):
    # loss هر نمونه در وزنش (بر اساس vote) ضرب میشه.
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        weights = inputs.pop("weight").float()
        outputs = model(**inputs)
        ce = torch.nn.CrossEntropyLoss(reduction="none")(outputs.logits, labels)
        loss = (ce * weights.to(ce.device)).sum() / weights.sum()
        return (loss, outputs) if return_outputs else loss

# سازگاری با نسخه‌های مختلف transformers
_ta_params = inspect.signature(TrainingArguments.__init__).parameters
EVAL_KW   = "eval_strategy" if "eval_strategy" in _ta_params else "evaluation_strategy"
WARMUP_KW = "warmup_ratio"  if "warmup_ratio"  in _ta_params else "warmup_steps"   # در نسخه‌ی جدید عدد اعشاری (<1) یعنی نسبت
TOK_KW    = "processing_class" if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer"

def run(name, use_vote=False, use_weights=False, epochs=EPOCHS):
    print(f"\n{'='*20} {name} {'='*20}")
    set_seed(SEED)

    train_ds = build_dataset(train_df, use_vote, use_weights, train=True)
    val_ds   = build_dataset(val_df,   use_vote, use_weights, train=False)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)

    args = TrainingArguments(
        output_dir=f"./out_{name}",
        num_train_epochs=epochs,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        learning_rate=lr,
        weight_decay=0.01,
        **{WARMUP_KW: 0.06},
        **{EVAL_KW: "epoch"},
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="micro_f1",
        save_total_limit=1,
        logging_steps=20,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        report_to="none",
        remove_unused_columns=not use_weights,   # برای حفظ ستون weight
    )

    TrainerCls = WeightedTrainer if use_weights else Trainer
    trainer = TrainerCls(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=DataCollatorWithPadding(tok),
        compute_metrics=compute_metrics,
        **{TOK_KW: tok},
    )
    trainer.train()

    out  = trainer.predict(val_ds)
    pred = out.predictions.argmax(-1)
    y    = val_df["label"].values
    micro = f1_score(y, pred, average="micro")
    macro = f1_score(y, pred, average="macro")

    print(f"\nمیکرو-F1: {micro:.4f} | ماکرو-F1: {macro:.4f}")
    print(classification_report(y, pred, target_names=["1","2","3","4","5"], zero_division=0))
    print("Confusion matrix (سطر: واقعی، ستون: پیش‌بینی):")
    print(confusion_matrix(y, pred))

    trainer.save_model(f"{SAVE_ROOT}/{name}")
    tok.save_pretrained(f"{SAVE_ROOT}/{name}")
    return micro, macro

In [ ]:
results = {}
results["text_only"]          = run("text_only")
results["with_vote_input"]    = run("with_vote_input",    use_vote=True)
results["vote_weighted_loss"] = run("vote_weighted_loss", use_weights=True)

summary = pd.DataFrame(results, index=["micro_F1", "macro_F1"]).T
print("\n", summary)
summary.to_csv(f"{SAVE_ROOT}/results_day2.csv")


==================== text_only ====================


Map:   0%|          | 0/4250 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

... (26 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


میکرو-F1: 0.6867 | ماکرو-F1: 0.4696
              precision    recall  f1-score   support

           1       0.57      0.81      0.67        74
           2       0.00      0.00      0.00        51
           3       0.39      0.38      0.38        72
           4       0.45      0.42      0.44       140
           5       0.83      0.89      0.86       413

    accuracy                           0.69       750
   macro avg       0.45      0.50      0.47       750
weighted avg       0.64      0.69      0.66       750

Confusion matrix (سطر: واقعی، ستون: پیش‌بینی):
[[ 60   1   9   1   3]
 [ 26   0  16   8   1]
 [ 10   0  27  25  10]
 [  6   0  14  59  61]
 [  4   0   3  37 369]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


                     micro_F1  macro_F1
text_only           0.692000  0.491435
with_vote_input     0.693333  0.504223
vote_weighted_loss  0.686667  0.469565


<div dir="rtl">

## مقایسه‌ی مدل‌های مختلف روی همون سمپل 5 هزارتایی

**هدف**: انتخاب معماری برای مدل نهایی. DistilBERT (سلول‌های بالا) مبنای مقایسه ماست. <br>

- همه‌ی مدل‌ها روی همون hyperparameter ها آموزش می‌بینن.
- ستون vote در این مقایسه دیگه استفاده نمیشه چون نتیجه‌ی تست‌های بالا معنادار نبود.
- چون مقدار ولیدیشن  فقط 750 ردیفه، اختلاف کمتر از حدود 3% micro-F1 رو «برتری» حساب نمیکنیم.

**قاعده‌ی تصمیم‌گیری**: اگر بهترین مدل بیش از حدود 3٪ از DistilBERT بهتر بود همون انتخاب میشه؛ در غیر این صورت مدل ساده‌تر و سریع‌تر ممونه.

</div>


In [ ]:
!pip install -q sentencepiece      # برای توکنایزر DeBERTa

In [ ]:
def run_model(name, model_name, lr=2e-5, epochs=EPOCHS, bs=32, fp16=True, seed=SEED):
    """آموزش یک مدل دلخواه روی train_df و ارزیابی روی val_df (فقط با متن full_text)."""
    global tok
    old_tok = tok                       # build_dataset از tok سراسری استفاده می‌کند
    print(f"\n{'='*20} {name} ({model_name}) {'='*20}")
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    try:
        train_ds = build_dataset(train_df)
        val_ds   = build_dataset(val_df, train=False)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)
        model = model.float()

        args = TrainingArguments(
            output_dir=f"./out_{name}",
            num_train_epochs=epochs,
            per_device_train_batch_size=bs,
            per_device_eval_batch_size=64,
            learning_rate=lr,
            weight_decay=0.01,
            **{WARMUP_KW: 0.06},
            **{EVAL_KW: "epoch"},
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="micro_f1",
            save_total_limit=1,
            logging_steps=50,
            fp16=fp16 and torch.cuda.is_available(),
            seed=seed,
            report_to="none",
        )
        trainer = Trainer(
            model=model, args=args,
            train_dataset=train_ds, eval_dataset=val_ds,
            data_collator=DataCollatorWithPadding(tok),
            compute_metrics=compute_metrics,
            **{TOK_KW: tok},
        )
        trainer.train()

        out  = trainer.predict(val_ds)
        pred = out.predictions.argmax(-1)
        y    = val_df["label"].values
        micro = f1_score(y, pred, average="micro")
        macro = f1_score(y, pred, average="macro")

        print(f"\nمیکرو-F1: {micro:.4f} | ماکرو-F1: {macro:.4f}")
        print(classification_report(y, pred, target_names=["1","2","3","4","5"], zero_division=0))
        print("Confusion matrix (سطر: واقعی، ستون: پیش‌بینی):")
        print(confusion_matrix(y, pred))

        trainer.save_model(f"{SAVE_ROOT}/{name}")
        tok.save_pretrained(f"{SAVE_ROOT}/{name}")
        np.save(f"{SAVE_ROOT}/{name}_val_logits.npy", out.predictions)   # برای تحلیل خطا
        return micro, macro
    finally:
        tok = old_tok                   # توکنایزر DistilBERT برای بقیه‌ی سلول‌ها برمی‌گردد


In [ ]:
# نتیجه‌ی DistilBERT از اجرای قبلی (همین split و همین تنظیمات)
results = {"distilbert (مبنا)": (0.6920, 0.4914)}

models = [
    ("roberta",       "roberta-base"),
    ("distilroberta", "distilroberta-base"),
    ("deberta",       "microsoft/deberta-v3-base"),
    ("modernbert",    "answerdotai/ModernBERT-base"),
    ("nlptown",       "nlptown/bert-base-multilingual-uncased-sentiment"),
]

for name, mid in models:
    try:
        try:
            results[name] = run_model(name, mid)
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            print(f"{name}: کمبود حافظه GPU، دوباره با batch size برابر 16")
            results[name] = run_model(name, mid, bs=16)
    except Exception as e:
        print(f"{name} ارور داد: {e}")
    gc.collect(); torch.cuda.empty_cache()

summary = (pd.DataFrame(results, index=["micro_F1", "macro_F1"]).T
             .sort_values("micro_F1", ascending=False))
print("\n", summary)
summary.to_csv(f"{SAVE_ROOT}/results_models.csv")



==================== roberta (roberta-base) ====================


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (73 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


میکرو-F1: 0.7240 | ماکرو-F1: 0.5929
              precision    recall  f1-score   support

           1       0.74      0.65      0.69        74
           2       0.46      0.33      0.39        51
           3       0.49      0.49      0.49        72
           4       0.52      0.54      0.53       140
           5       0.86      0.89      0.87       413

    accuracy                           0.72       750
   macro avg       0.61      0.58      0.59       750
weighted avg       0.72      0.72      0.72       750

Confusion matrix (سطر: واقعی، ستون: پیش‌بینی):
[[ 48  12   6   3   5]
 [ 11  17  15   5   3]
 [  5   5  35  20   7]
 [  1   3  13  76  47]
 [  0   0   3  43 367]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


                    micro_F1  macro_F1
roberta            0.733333  0.583432
modernbert         0.728000  0.571221
distilroberta      0.724000  0.544003
nlptown            0.724000  0.592895
distilbert (مبنا)  0.692000  0.491400


In [ ]:
results["deberta"] = run_model("deberta", "microsoft/deberta-v3-base")

summary = (pd.DataFrame(results, index=["micro_F1", "macro_F1"]).T
             .sort_values("micro_F1", ascending=False))
print(summary)
summary.to_csv(f"{SAVE_ROOT}/results_models.csv")


==================== deberta (microsoft/deberta-v3-base) ====================


Map:   0%|          | 0/4250 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

... (4 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


میکرو-F1: 0.7267 | ماکرو-F1: 0.5508
              precision    recall  f1-score   support

           1       0.66      0.82      0.73        74
           2       0.40      0.20      0.26        51
           3       0.37      0.29      0.33        72
           4       0.53      0.55      0.54       140
           5       0.87      0.91      0.89       413

    accuracy                           0.73       750
   macro avg       0.57      0.55      0.55       750
weighted avg       0.71      0.73      0.71       750

Confusion matrix (سطر: واقعی، ستون: پیش‌بینی):
[[ 61   5   4   3   1]
 [ 19  10  18   3   1]
 [  8   9  21  31   3]
 [  3   1  10  77  49]
 [  1   0   4  32 376]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

                   micro_F1  macro_F1
roberta            0.733333  0.583432
modernbert         0.728000  0.571221
deberta            0.726667  0.550839
distilroberta      0.724000  0.544003
nlptown            0.724000  0.592895
distilbert (مبنا)  0.692000  0.491400


In [ ]:
from scipy.special import softmax

names = ["roberta", "deberta", "modernbert", "nlptown", "distilroberta"]
P = {n: softmax(np.load(f"{SAVE_ROOT}/{n}_val_logits.npy"), axis=1) for n in names}
y = val_df["label"].values

def score(ns):
    p = sum(P[n] for n in ns).argmax(1)
    return round(f1_score(y, p, average="micro"), 4), round(f1_score(y, p, average="macro"), 4)

for c in [["roberta", "deberta"],
          ["roberta", "modernbert", "nlptown"],
          ["roberta", "deberta", "modernbert", "nlptown"],
          names]:
    print(c, score(c))

['roberta', 'deberta'] (0.7413, 0.5773)
['roberta', 'modernbert', 'nlptown'] (0.7333, 0.5815)
['roberta', 'deberta', 'modernbert', 'nlptown'] (0.748, 0.5913)
['roberta', 'deberta', 'modernbert', 'nlptown', 'distilroberta'] (0.744, 0.5802)


In [ ]:
import zipfile, glob
from scipy.special import softmax
from sklearn.metrics import log_loss

# ---------- 1) یک validation بزرگ و تازه از فایل خام train ----------
if not os.path.exists("/content/raw/train"):
    os.makedirs("/content/raw", exist_ok=True)
    gdown.download(id="1Dzp8Ks1aQoscJA6XLRXg-OtqunshlMfJ", output="/content/raw/train.zip", quiet=False)
    zipfile.ZipFile("/content/raw/train.zip").extractall("/content/raw/train")
TRAIN_PATH = max(glob.glob("/content/raw/train/**/*.csv", recursive=True), key=os.path.getsize)

big = pd.read_csv(TRAIN_PATH, usecols=["overall", "full_text"])
seen = set(train_df["full_text"]) | set(val_df["full_text"])
big = big[~big["full_text"].isin(seen)].drop_duplicates(subset=["full_text"])   # بدون نشت از نمونه‌ی ۵ هزارتایی
val_big = big.sample(n=5000, random_state=123).reset_index(drop=True)
val_big["label"] = val_big["overall"].astype(int) - 1
y = val_big["label"].values
print("val_big:", val_big.shape)
print(val_big["overall"].value_counts(normalize=True).sort_index().round(3).to_dict())
del big

# ---------- 2) ارزیابی مدل‌های ذخیره‌شده (بدون آموزش دوباره) ----------
def predict_logits(name, texts, bs=128):
    path = f"{SAVE_ROOT}/{name}"
    tk = AutoTokenizer.from_pretrained(path)
    m  = AutoModelForSequenceClassification.from_pretrained(path).to("cuda").eval()
    order = np.argsort([len(t) for t in texts])            # مرتب‌سازی بر اساس طول برای سرعت
    out = np.zeros((len(texts), 5), dtype=np.float32)
    for i in range(0, len(texts), bs):
        idx = order[i:i+bs]
        enc = tk([texts[j] for j in idx], truncation=True, max_length=MAX_LEN,
                 padding=True, return_tensors="pt").to("cuda")
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            out[idx] = m(**enc).logits.float().cpu().numpy()
    del m; gc.collect(); torch.cuda.empty_cache()
    return out

saved = {"distilbert": "text_only", "roberta": "roberta", "distilroberta": "distilroberta",
         "deberta": "deberta", "modernbert": "modernbert", "nlptown": "nlptown"}

texts = val_big["full_text"].tolist()
P = {}
for label, folder in saved.items():
    try:
        P[label] = softmax(predict_logits(folder, texts), axis=1)
        np.save(f"{SAVE_ROOT}/{folder}_valbig_probs.npy", P[label])
    except Exception as e:
        print(f"{label} ارور داد: {e}")

# ---------- 3) جدول مقایسه ----------
def row(p):
    pred = p.argmax(1)
    return (f1_score(y, pred, average="micro"), f1_score(y, pred, average="macro"),
            log_loss(y, p, labels=range(5)))

rows = {k: row(p) for k, p in P.items()}
ens = {"ens: roberta+deberta": ["roberta", "deberta"],
       "ens: roberta+deberta+modernbert+nlptown": ["roberta", "deberta", "modernbert", "nlptown"]}
for k, ns in ens.items():
    if all(n in P for n in ns):
        rows[k] = row(sum(P[n] for n in ns) / len(ns))

res = pd.DataFrame(rows, index=["micro_F1", "macro_F1", "log_loss"]).T.sort_values("micro_F1", ascending=False)
res["±95%"] = 1.96 * np.sqrt(res["micro_F1"] * (1 - res["micro_F1"]) / len(y))
print(res.round(4))
res.to_csv(f"{SAVE_ROOT}/results_valbig.csv")

val_big: (5000, 3)
{1: 0.097, 2: 0.067, 3: 0.088, 4: 0.181, 5: 0.566}


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

                                         micro_F1  macro_F1  log_loss    ±95%
ens: roberta+deberta+modernbert+nlptown    0.7404    0.5752    0.6368  0.0122
ens: roberta+deberta                       0.7330    0.5512    0.6647  0.0123
deberta                                    0.7290    0.5463    0.6813  0.0123
nlptown                                    0.7284    0.5844    0.6935  0.0123
roberta                                    0.7226    0.5532    0.7101  0.0124
distilroberta                              0.7026    0.5215    0.7539  0.0127
modernbert                                 0.6990    0.5355    0.7786  0.0127
distilbert                                 0.6824    0.4625    0.7867  0.0129
